In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
torch.set_printoptions(precision=2, sci_mode=False, linewidth=120)

print("torch", torch.__version__)

torch 2.14.1+cpu


In [3]:
def show(mat, title="", row_labels=None, col_labels=None, decimals=2):
    """Print a 2-D tensor with token names on the rows and columns."""
    row_labels = row_labels or TOKENS
    col_labels = col_labels or TOKENS
    if title:
        print(title)
    print("       " + "".join(f"{c:>8}" for c in col_labels))
    for name, row in zip(row_labels, mat.tolist()):
        body = "".join(f"{val:8.{decimals}f}" for val in row)
        print(f"{name:>6} {body}")
    print()

In [4]:
TOKENS = ["the", "cat", "sat", "on", "mat"]

B = 1
T = len(TOKENS)
d_model = 8
n_heads = 2
d_k = d_model // n_heads

In [5]:
x = torch.randn(B, T, d_model)

In [6]:
x.shape, x

(torch.Size([1, 5, 8]),
 tensor([[[-1.13, -1.15, -0.25, -0.43,  0.85,  0.69, -0.32, -2.12],
          [ 0.32, -1.26,  0.35,  0.31,  0.12,  1.24,  1.12, -0.25],
          [-1.35, -1.70,  0.57,  0.79,  0.60, -1.56, -0.34,  1.85],
          [-0.22, -0.74,  0.56,  0.26, -0.17, -0.68,  0.94,  0.49],
          [ 1.20,  0.08, -1.20, -0.00, -0.52, -0.31, -1.58,  1.71]]]))

---

In [7]:
naive_scores = x[0] @ x[0].T
show(naive_scores)

            the     cat     sat      on     mat
   the     8.62    2.00   -1.39   -1.10   -4.91
   cat     2.00    4.77   -0.54    1.21   -2.77
   sat    -1.39   -0.54   11.98    3.61    1.41
    on    -1.10    1.21    3.61    2.59   -1.35
   mat    -4.91   -2.77    1.41   -1.35    8.67



In [8]:
W_q = nn.Linear(d_model, d_model, bias=False)
W_k = nn.Linear(d_model, d_model, bias=False)
W_v = nn.Linear(d_model, d_model, bias=False)

q = W_q(x)
k = W_k(x)
v = W_v(x)

In [9]:
print(x[0][2])

tensor([-1.35, -1.70,  0.57,  0.79,  0.60, -1.56, -0.34,  1.85])


In [10]:
print(q[0][2])

tensor([ 0.24, -0.66, -0.16, -0.36,  0.03, -0.97,  0.44, -0.29], grad_fn=<SelectBackward0>)


In [11]:
print(k[0][2])

tensor([-0.35,  0.41, -0.05,  0.72, -0.07,  0.05,  1.05,  0.49], grad_fn=<SelectBackward0>)


In [12]:
print(v[0][2])

tensor([-0.67, -0.24, -0.66, -0.07,  0.04,  0.44,  0.08, -0.66], grad_fn=<SelectBackward0>)


In [13]:
scores = q @ k.transpose(-2, -1)

In [14]:
show(scores[0])

            the     cat     sat      on     mat
   the     0.70   -1.33    0.45   -0.22    0.33
   cat    -0.69   -0.17   -1.03   -0.20    0.78
   sat    -0.51   -0.31   -0.34   -0.35    0.07
    on    -0.45    0.47   -0.55   -0.03   -0.27
   mat    -0.87   -0.58   -0.46   -0.53    1.74



In [15]:
scaled = scores / d_model ** 0.5

In [16]:
show(scaled[0])

            the     cat     sat      on     mat
   the     0.25   -0.47    0.16   -0.08    0.12
   cat    -0.24   -0.06   -0.36   -0.07    0.27
   sat    -0.18   -0.11   -0.12   -0.12    0.03
    on    -0.16    0.17   -0.19   -0.01   -0.10
   mat    -0.31   -0.21   -0.16   -0.19    0.62



In [17]:
max_seq_len = 5

sent = torch.ones(max_seq_len, max_seq_len, dtype=torch.bool)
mask = torch.tril(sent)

show(sent)

            the     cat     sat      on     mat
   the     1.00    1.00    1.00    1.00    1.00
   cat     1.00    1.00    1.00    1.00    1.00
   sat     1.00    1.00    1.00    1.00    1.00
    on     1.00    1.00    1.00    1.00    1.00
   mat     1.00    1.00    1.00    1.00    1.00



In [18]:
show(mask)

            the     cat     sat      on     mat
   the     1.00    0.00    0.00    0.00    0.00
   cat     1.00    1.00    0.00    0.00    0.00
   sat     1.00    1.00    1.00    0.00    0.00
    on     1.00    1.00    1.00    1.00    0.00
   mat     1.00    1.00    1.00    1.00    1.00



In [19]:
mask = torch.tril(
    torch.ones(
        T, T, dtype=torch.bool
    )
)

In [20]:
show(mask)

            the     cat     sat      on     mat
   the     1.00    0.00    0.00    0.00    0.00
   cat     1.00    1.00    0.00    0.00    0.00
   sat     1.00    1.00    1.00    0.00    0.00
    on     1.00    1.00    1.00    1.00    0.00
   mat     1.00    1.00    1.00    1.00    1.00



In [21]:
masked = scaled.masked_fill(~mask, float("-inf"))
show(masked[0])

            the     cat     sat      on     mat
   the     0.25    -inf    -inf    -inf    -inf
   cat    -0.24   -0.06    -inf    -inf    -inf
   sat    -0.18   -0.11   -0.12    -inf    -inf
    on    -0.16    0.17   -0.19   -0.01    -inf
   mat    -0.31   -0.21   -0.16   -0.19    0.62



In [22]:
weights = F.softmax(masked, dim=-1)
out = weights @ v

In [23]:
show(out[0])

            the     cat     sat      on     mat
   the    -1.00    0.40   -0.70   -0.28    0.74   -0.51   -0.58    0.05
   cat    -0.29   -0.01   -0.53   -0.21    0.73   -0.55   -0.74   -0.19
   sat    -0.44   -0.07   -0.58   -0.16    0.50   -0.21   -0.46   -0.34
    on    -0.24   -0.17   -0.54   -0.12    0.44   -0.20   -0.41   -0.25
   mat    -0.05    0.04    0.05    0.01   -0.09    0.09   -0.15   -0.38



In [31]:
def single_head_attention(x: torch.Tensor, W_q: torch.nn.Linear, W_k: torch.nn.Linear, W_v: torch.nn.Linear, mask: torch.Tensor) -> torch.Tensor:

    B, T, d_model = x.shape

    q = W_q(x)
    k = W_k(x)
    v = W_v(x)

    q_k = q @ k.transpose(-2, -1)

    q_k_hat = q_k / (d_model ** 0.5)

    causal = mask[:T, :T]
    masked = q_k_hat.masked_fill(~causal, float("-inf"))

    weights = F.softmax(masked, dim=-1)

    out = weights @ v

    return out, weights

In [32]:
head_out, head_weight = single_head_attention(x, W_q, W_k, W_v, mask)

In [34]:
print(f"in  : {x.shape}")
print(f"out : {head_out.shape}")

in  : torch.Size([1, 5, 8])
out : torch.Size([1, 5, 8])


In [ ]:
head_a, head_b = None, None

torch.manual_seed(2)
head_a = [
    nn.Linear(d_model, d_model, bias=False)
    for _ in range(3)
]

torch.manual_seed(4)
head_b = [
    nn.Linear(d_model, d_model, bias=False)
    for _ in range(3)
]

o_a, w_a = single_head_attention(x, *head_a, mask)
o_b, w_b = single_head_attention(x, *head_b, mask)

print(f"head_a: \n{o_a}\n\n{w_a}\n")
print(f"head_b: \n{o_b}\n\n{w_b}")

head_a: 
tensor([[[-0.01, -0.61, -0.06, -0.08,  0.40,  0.81, -0.29,  1.56],
         [-0.04,  0.07, -0.29,  0.01, -0.05,  0.28, -0.12,  1.04],
         [ 0.54,  0.31, -0.20,  0.03, -0.29,  0.48,  0.20,  0.70],
         [ 0.54,  0.40, -0.19,  0.11, -0.34,  0.31,  0.26,  0.54],
         [ 0.58,  0.25, -0.09, -0.02, -0.29,  0.45,  0.11,  0.12]]], grad_fn=<UnsafeViewBackward0>)

tensor([[[1.00, 0.00, 0.00, 0.00, 0.00],
         [0.46, 0.54, 0.00, 0.00, 0.00],
         [0.26, 0.39, 0.35, 0.00, 0.00],
         [0.22, 0.27, 0.23, 0.29, 0.00],
         [0.17, 0.14, 0.27, 0.18, 0.24]]], grad_fn=<SoftmaxBackward0>)

head_b: 
tensor([[[-0.34, -0.21, -0.43,  0.31, -0.75,  0.48,  0.38, -0.16],
         [-0.23, -0.12, -0.31,  0.22, -0.37,  0.46,  0.05, -0.26],
         [-0.14, -0.13, -0.30,  0.00,  0.06,  0.81, -0.01,  0.20],
         [-0.12, -0.17, -0.32,  0.04, -0.00,  0.62, -0.07,  0.07],
         [-0.09, -0.02, -0.13, -0.10,  0.33,  0.57, -0.00,  0.30]]], grad_fn=<UnsafeViewBackward0>)

tensor([

In [39]:
show(w_a[0], "head A")
show(w_b[0], "head B")

head A
            the     cat     sat      on     mat
   the     1.00    0.00    0.00    0.00    0.00
   cat     0.46    0.54    0.00    0.00    0.00
   sat     0.26    0.39    0.35    0.00    0.00
    on     0.22    0.27    0.23    0.29    0.00
   mat     0.17    0.14    0.27    0.18    0.24

head B
            the     cat     sat      on     mat
   the     1.00    0.00    0.00    0.00    0.00
   cat     0.49    0.51    0.00    0.00    0.00
   sat     0.28    0.36    0.36    0.00    0.00
    on     0.31    0.30    0.15    0.24    0.00
   mat     0.15    0.17    0.30    0.19    0.19



In [83]:
def multi_head_attention(
        x: torch.Tensor, 
        W_q: torch.nn.Linear, W_k: torch.nn.Linear, W_v: torch.nn.Linear, W_o: torch.Tensor, 
        mask: torch.Tensor, 
        n_heads: int,
    ) -> torch.Tensor:

    B, T, d_model = x.shape

    d_k = d_model // n_heads

    q = W_q(x).view(B, T, n_heads, d_k).transpose(1, 2)
    k = W_k(x).view(B, T, n_heads, d_k).transpose(1, 2)
    v = W_v(x).view(B, T, n_heads, d_k).transpose(1, 2)

    q_k = q @ k.transpose(-2, -1)

    q_k_hat = q_k / (d_model ** 0.5)

    causal = mask[:T, :T]
    masked = q_k_hat.masked_fill(~causal, float("-inf"))

    weights = F.softmax(masked, dim=-1)

    out = weights @ v

    out_hat = out.transpose(2, 1).contiguous().view(B, T, d_model)
    o = W_o(out_hat)

    return o, weights

In [ ]:
W_o = nn.Linear(d_model, d_model)

mha_out, mha_w = multi_head_attention(x, W_q, W_k, W_v, W_o, mask, n_heads)



(tensor([[[ 0.04,  0.21,  0.09, -0.18,  0.15, -0.76,  0.09, -0.15],
          [ 0.05,  0.03,  0.15, -0.30,  0.16, -0.56, -0.05, -0.22],
          [ 0.11,  0.07,  0.11, -0.21,  0.26, -0.39,  0.16, -0.18],
          [ 0.05,  0.03,  0.11, -0.19,  0.23, -0.32,  0.15, -0.18],
          [-0.12, -0.15, -0.06, -0.43,  0.29, -0.24,  0.21, -0.04]]], grad_fn=<ViewBackward0>),
 tensor([[[[1.00, 0.00, 0.00, 0.00, 0.00],
           [0.44, 0.56, 0.00, 0.00, 0.00],
           [0.33, 0.35, 0.33, 0.00, 0.00],
           [0.25, 0.27, 0.24, 0.25, 0.00],
           [0.14, 0.19, 0.16, 0.20, 0.31]],
 
          [[1.00, 0.00, 0.00, 0.00, 0.00],
           [0.52, 0.48, 0.00, 0.00, 0.00],
           [0.32, 0.33, 0.35, 0.00, 0.00],
           [0.22, 0.29, 0.22, 0.26, 0.00],
           [0.21, 0.17, 0.22, 0.17, 0.24]]]], grad_fn=<SoftmaxBackward0>))